<a href="https://colab.research.google.com/github/DABMASTER-Brought-me-into-this/ForFunModel/blob/main/MyCustomModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Neural Network
When hypothesizing how Transformers are able to take a variable # of inputs (from 0 - context length) I predicted that they did so by considering every single combination of the tokens prior to it and did inference on that. While this model is completely unrealistic to be used with subword emebeddings, I am very curious on how it will fair with char level embeddings. I am also curious if I change the language to something other language with fewer characters (ex: Rotokas, or Toki Pona) as they have less characters.

In [ ]:
# Dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-06-30 17:44:49--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.2’

input.txt.2         100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-06-30 17:44:49 (20.6 MB/s) - ‘input.txt.2’ saved [1115394/1115394]



In [ ]:
# Standrad Imports
import re
import math
import torch # Using Torch J for This One
import random
import itertools
import torch.nn.functional as F

In [ ]:
# Hyper Parameter
context_length = 8
N_EMBD = 10
NUMBER_OF_NEURONS_L1 = 16
NUMBER_OF_NEURONS_L2 = 8

In [ ]:
# Opening Dataset + Dataset Cleansing
with open('input.txt', 'r') as file:
  text = file.read().lower()
  text = re.sub('[^a-z ]', '', text)

In [ ]:
# Creating Mappings of tokens to string and vice versa
stoi = {letter: i for i, letter in enumerate(sorted(list(set(text))))}
itos = dict(zip(stoi.values(), stoi.keys()))

In [ ]:
# Tokenizing the Dataset
tokens = list(map(lambda input: stoi[input], list(text)))

In [ ]:
# Creating the Inputs
inputs = []

In [ ]:
# Creating "Expanded" Dataset
for i in range(0, len(tokens[:10]), context_length):
  for j in range(context_length):
    combinations = list(itertools.combinations(itos.keys(), context_length-j-1))
    expanded_inputs = [list(combinations) + tokens[i:j+1+i] for combinations in combinations]
    inputs.extend(expanded_inputs)

In [ ]:
input = torch.tensor(inputs)

In [ ]:
from functools import lru_cache
### Creating PyTorch-Ish Classes
class Embedding:
  def __init__(self, vocab_size, n_embd):
    self.C = torch.randn(vocab_size, n_embd, requires_grad=True)

  def __call__(self, X):
    self.X = X
    self.out = self.C[self.X]
    return self.out

  def _parameters(self):
    return [self.C]

# ====================================================================================================

class Conv1D:
  def __call__(self, X):
    self.X = X

    if self.X.ndim > 2:
      self.out = self.X.view(-1, self.X.shape[1]//2, self.X.shape[2] * 2)
      if self.X.shape[1] == 1:
        self.out = torch.squeeze(self.out, dim = 1)
    else:
      assert True, f"ERROR: X has {self.X.ndim} dimensions"

    return self.out

# ====================================================================================================

class Linear:
  def __init__(self, fan_in, fan_out, bias=False):
    self.W = torch.randn(fan_in, fan_out, requires_grad=True) * 0.01
    self.B = torch.randn(fan_out, requires_grad=True) if bias else None

  def __call__(self, X):
    self.X = X
    self.out = self.X @ self.W
    if self.B is not None:
      self.out += self.B
    return self.out

  def _parameters(self):
    return [self.W, self.B] if self.B is not None else [self.W]

# ====================================================================================================

class BatchNorm:
  def __init__(self, fan_out, training = True):
    self.g = torch.ones(fan_out, requires_grad=True)
    self.b = torch.zeros(fan_out, requires_grad=True)
    self.running_mean = None
    self.running_var = None
    self.training = training

  def __call__(self, X):
    self.X = X

    if self.training:
      self.xmean = self.X.mean(axis = 0, keepdim=True)
      self.xvar = self.X.var(axis = 0, keepdim=True)

      if self.running_mean is None or self.running_var is None:
        self.running_mean = self.xmean
        self.running_var = self.xvar
      else:
        self.running_mean = self.running_mean * 0.9 + self.xmean * 0.1
        self.running_var = self.running_var * 0.9 + self.xvar * 0.1

    self.prebn = (self.X - self.xmean)/torch.sqrt(self.xvar + 1e-5)
    self.out = self.g * self.prebn + self.b
    return self.out

  def _parameters(self):
    return [self.g, self.b]

# ====================================================================================================

class ReLu:
  def __call__(self, X):
    self.X = X
    self.out = torch.maximum(torch.zeros_like(self.X), self.X)
    return self.out

  def _parameters(self):
    return []

# ====================================================================================================

class SoftMax:
  def __call__(self, X):
    self.X = X
    self.logits -= torch.max(self.X, dim=0, keepdim=True)
    self.probs = torch.e ** self.logits
    self.out = self.probs
    return self.out

  def _parameters(self):
    return []

# ====================================================================================================

class nll:
  def __call__(self, probs, targets):
    self.loss = -torch.log(probs[targets])
    self.out = self.loss
    return self.out
# ====================================================================================================

class Model:
  def __init__(self, layers, loss_func, lr):
    self.layers = layers
    self.loss_func = loss_func
    self.parameters = [parameter for layer in self.layers for parameter in layer._parameters()]
    self.lr = lr

  def forward(self, X):
    self.out = X
    for layer in self.layers:
      self.out = layer(self.out)

    self.loss = self.loss_func(self.out)
    return self.loss

  def backwards(self):
    self.loss.backward()

  def update_parameters(self):
    for parameter in self.parameters:
      parameter.data -= parameter.grad * self.lr

In [ ]:
C = torch.randn(len(stoi), N_EMBD)
C[input].shape

torch.Size([2571248, 8, 10])